# 2. ReAct Agent vs. Flat Retrieval: Multi-Hop Questions over a Real ServiceNow Incident Graph

**Presentation map:** Slide 20 (the ReAct loop with a knowledge graph as the environment), Slides 40-42 (traversal lifts
multi-hop QA), Slide 43 (exposing a graph to an agent as tools), Slides 44 & 61 (templated, read-only, parameterised
queries instead of free-form query generation)

**Dataset:** the real UCI ServiceNow incident-management event log in the course warehouse
`/Users/dt/cc/AI and Data Utilization - WS/data/course.duckdb` (opened **read-only**):
- `incidents.incident_events` — **119,998** field-change snapshots
- `incidents.incident_summary` — **20,769** incidents (one row each: priority, SLA, time-to-resolve)

Every incident that references a **problem record**, a **change request** (`rfc` / `caused_by`) becomes a node in an
operational graph, linked to its problem, change, assignment group, category, priority and SLA outcome — **all edges
come from real warehouse columns**; nothing is synthetic.

The experiment: 10 multi-hop questions whose gold answers are computed independently with SQL. We compare
**(a)** a ReAct agent that can only *traverse* the graph through a few templated, read-only tools against
**(b)** a single-shot "flat RAG" baseline that sees the top-10 most similar incident rows.

In [1]:
import json, re
from collections import Counter, defaultdict
import duckdb
import networkx as nx

DUCKDB_PATH = "/Users/dt/cc/AI and Data Utilization - WS/data/course.duckdb"
con = duckdb.connect(DUCKDB_PATH, read_only=True)   # guardrail #1: the connection itself cannot write

n_events = con.execute("SELECT count(*) FROM incidents.incident_events").fetchone()[0]
n_incidents = con.execute("SELECT count(*) FROM incidents.incident_summary").fetchone()[0]
print(f"Warehouse: {n_events:,} incident_events rows, {n_incidents:,} incidents")

# One row per incident: links ever recorded on any snapshot (max = the non-null value), latest category/group,
# plus SLA/priority facts from the summary table. Keep incidents that have at least one problem/change link.
con.execute("""
CREATE TEMP TABLE linked AS
WITH e AS (
    SELECT number,
           max(problem_id) AS problem_id, max(rfc) AS rfc, max(caused_by) AS caused_by,
           arg_max(category, sys_updated_at) AS category,
           arg_max(assignment_group, sys_updated_at) AS assignment_group
    FROM incidents.incident_events GROUP BY number)
SELECT e.*, s.priority, s.made_sla, s.ttr_hours
FROM e JOIN incidents.incident_summary s ON s.incident = e.number
WHERE e.problem_id IS NOT NULL OR e.rfc IS NOT NULL OR e.caused_by IS NOT NULL
""")
rows = con.execute("SELECT * FROM linked").fetchall()
cols = [d[0] for d in con.description]
records = [dict(zip(cols, r)) for r in rows]

norm = lambda s: re.sub(r"\s+", " ", s).strip() if isinstance(s, str) else s   # 'Problem ID  8' -> 'Problem ID 8'

G = nx.MultiDiGraph()
def link(src, rel, dst, label):
    if dst is None:
        return
    dst = norm(dst)
    G.add_node(dst, label=label)
    G.add_edge(src, dst, key=rel, type=rel)

for r in records:
    G.add_node(r["number"], label="Incident", priority=r["priority"], made_sla=bool(r["made_sla"]),
               ttr_hours=round(r["ttr_hours"] or 0, 1), category=r["category"], assignment_group=r["assignment_group"])
    link(r["number"], "LINKED_TO_PROBLEM", r["problem_id"], "Problem")
    link(r["number"], "RELATED_CHANGE", r["rfc"], "ChangeRequest")
    link(r["number"], "CAUSED_BY_CHANGE", r["caused_by"], "ChangeRequest")
    link(r["number"], "ASSIGNED_TO", r["assignment_group"], "AssignmentGroup")
    link(r["number"], "IN_CATEGORY", r["category"], "Category")
    link(r["number"], "HAS_PRIORITY", r["priority"], "Priority")
    link(r["number"], "SLA_OUTCOME", "SLA met" if r["made_sla"] else "SLA breached", "SLAOutcome")

print(f"\nOperational graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges (all from warehouse columns)")
print("Nodes by label:", dict(Counter(nx.get_node_attributes(G, "label").values())))
print("Edges by type: ", dict(Counter(d["type"] for _, _, d in G.edges(data=True))))

Warehouse: 119,998 incident_events rows, 20,769 incidents

Operational graph: 893 nodes, 2310 edges (all from warehouse columns)
Nodes by label: {'Incident': 456, 'Problem': 208, 'AssignmentGroup': 43, 'Category': 22, 'Priority': 4, 'SLAOutcome': 2, 'ChangeRequest': 158}
Edges by type:  {'LINKED_TO_PROBLEM': 325, 'ASSIGNED_TO': 456, 'IN_CATEGORY': 455, 'HAS_PRIORITY': 456, 'SLA_OUTCOME': 456, 'RELATED_CHANGE': 160, 'CAUSED_BY_CHANGE': 2}


## Templated, read-only graph tools (Slides 43, 44, 61)

The agent never writes a query. It gets four **parameterised templates** — the "fixed Cypher template whose parameters
the LLM fills" pattern from Slide 43. `traverse` is a generic path template (`start -[r1]- x -[r2]- y ...`), and
`category_sla_stats` is a fixed SQL aggregate.

It also fixes a bug in the previous version of this notebook: joining `incident_summary` to `incident_events` without
de-duplication counted **event rows** (3,271 for Category 45) instead of incidents.

In [2]:
def _resolve(node_id):
    node_id = norm(node_id)
    return node_id if node_id in G else None

def get_node(node_id: str) -> str:
    """Return the type and attributes of one graph node (Incident, Problem, ChangeRequest, AssignmentGroup, Category, Priority, SLAOutcome)."""
    n = _resolve(node_id)
    return json.dumps({"id": n, **G.nodes[n]} if n else {"error": f"no such node: {node_id}"})

def get_neighbors(node_id: str, relation: str = None, limit: int = 25) -> str:
    """List neighbours of a node (both directions), optionally filtered by relation type; returns the total count and up to `limit` neighbours."""
    n = _resolve(node_id)
    if not n:
        return json.dumps({"error": f"no such node: {node_id}"})
    nbrs = [(d["type"], v) for _, v, d in G.out_edges(n, data=True)] + [(d["type"], u) for u, _, d in G.in_edges(n, data=True)]
    if relation:
        nbrs = [x for x in nbrs if x[0] == relation]
    by_rel = Counter(r for r, _ in nbrs)
    return json.dumps({"id": n, "label": G.nodes[n]["label"], "counts_by_relation": by_rel,
                       "neighbors": [{"relation": r, "id": v, "label": G.nodes[v]["label"]} for r, v in nbrs[:limit]]})

def traverse(start_id: str, relations: list) -> str:
    """Path template: follow the given chain of relation types from start_id (direction ignored).
    Returns every distinct endpoint with the number of distinct nodes at the previous hop that reach it."""
    n = _resolve(start_id)
    if not n:
        return json.dumps({"error": f"no such node: {start_id}"})
    frontier = {n}
    for rel in relations:
        reach = defaultdict(set)
        for x in frontier:
            for _, v, d in G.out_edges(x, data=True):
                if d["type"] == rel: reach[v].add(x)
            for u, _, d in G.in_edges(x, data=True):
                if d["type"] == rel: reach[u].add(x)
        frontier = set(reach)
    last = reach  # endpoints of the final hop -> the nodes they were reached from
    endpoints = sorted(((v, len(src)) for v, src in last.items()), key=lambda t: -t[1])
    return json.dumps({"start": n, "path": relations, "n_endpoints": len(endpoints), "start_included": n in last,
                       "endpoints": [{"id": v, "label": G.nodes[v]["label"], "reached_from": k} for v, k in endpoints[:40]]})

def category_sla_stats(category: str) -> str:
    """Fixed read-only SQL aggregate over ALL warehouse incidents of a category: incident count, SLA breach rate, mean time-to-resolve."""
    row = con.execute("""
        SELECT count(*), avg(1 - s.made_sla::INT), avg(s.ttr_hours)
        FROM incidents.incident_summary s
        WHERE s.incident IN (SELECT DISTINCT number FROM incidents.incident_events WHERE category = ?)
    """, [category]).fetchone()
    return json.dumps({"category": category, "n_incidents": row[0],
                       "sla_breach_rate": round(row[1], 3) if row[1] is not None else None,
                       "avg_ttr_hours": round(row[2], 1) if row[2] is not None else None})

TOOL_IMPL = {f.__name__: f for f in (get_node, get_neighbors, traverse, category_sla_stats)}
TOOLS = [
    {"type": "function", "function": {"name": "get_node", "description": get_node.__doc__,
        "parameters": {"type": "object", "properties": {"node_id": {"type": "string"}}, "required": ["node_id"]}}},
    {"type": "function", "function": {"name": "get_neighbors", "description": get_neighbors.__doc__,
        "parameters": {"type": "object", "properties": {"node_id": {"type": "string"},
                       "relation": {"type": "string"}, "limit": {"type": "integer"}}, "required": ["node_id"]}}},
    {"type": "function", "function": {"name": "traverse", "description": traverse.__doc__,
        "parameters": {"type": "object", "properties": {"start_id": {"type": "string"},
                       "relations": {"type": "array", "items": {"type": "string",
                           "enum": sorted({d["type"] for _, _, d in G.edges(data=True)})}}},
                       "required": ["start_id", "relations"]}}},
    {"type": "function", "function": {"name": "category_sla_stats", "description": category_sla_stats.__doc__,
        "parameters": {"type": "object", "properties": {"category": {"type": "string"}}, "required": ["category"]}}},
]

print("Bug fix check — Category 45:")
buggy = con.execute("""SELECT count(*) FROM incidents.incident_summary s
                       JOIN incidents.incident_events e ON s.incident = e.number WHERE e.category = 'Category 45'""").fetchone()[0]
print(f"  old join (counts event rows): {buggy:,}   |   fixed tool: {category_sla_stats('Category 45')}")
print("\nExample path template — incidents sharing a change with INC0000388:")
print(" ", traverse("INC0000388", ["RELATED_CHANGE", "RELATED_CHANGE"]))

Bug fix check — Category 45:
  old join (counts event rows): 3,271   |   fixed tool: {"category": "Category 45", "n_incidents": 597, "sla_breach_rate": 0.764, "avg_ttr_hours": 529.8}

Example path template — incidents sharing a change with INC0000388:
  {"start": "INC0000388", "path": ["RELATED_CHANGE", "RELATED_CHANGE"], "n_endpoints": 2, "start_included": true, "endpoints": [{"id": "INC0000388", "label": "Incident", "reached_from": 1}, {"id": "INC0002325", "label": "Incident", "reached_from": 1}]}


## Guardrail demo: why the agent gets templates, not a free-form SQL tool (Slides 44, 61)

If we *did* expose a free-form query tool, two layers would have to stop a write: a statement-type check
(the DuckDB analogue of `EXPLAIN`-classifying a Cypher query in `mcp-neo4j-cypher`) and the read-only connection itself.

In [3]:
def guarded_sql(query: str):
    """Free-form SQL, accepted only if the parser classifies every statement as a SELECT."""
    stmts = duckdb.extract_statements(query)
    kinds = [s.type.name for s in stmts]
    if any(k != "SELECT" for k in kinds):
        return {"refused": True, "reason": f"statement types {kinds} are not read-only"}
    return {"refused": False, "rows": con.execute(query).fetchall()[:3]}

print("1)", guarded_sql("SELECT priority, count(*) FROM incidents.incident_summary GROUP BY 1"))
print("2)", guarded_sql("DELETE FROM incidents.incident_summary WHERE priority = '4 - Low'"))
print("3)", guarded_sql("SELECT 1; DROP TABLE incidents.incident_events"))
try:  # even if the guard were bypassed, the read-only connection rejects the write
    con.execute("INSERT INTO incidents.incident_events (number) VALUES ('INC9999999')")
except Exception as e:
    print("4) read-only connection:", type(e).__name__, "-", str(e).splitlines()[0][:90])

1) {'refused': False, 'rows': [('4 - Low', 658), ('2 - High', 337), ('3 - Moderate', 19554)]}
2) {'refused': True, 'reason': "statement types ['DELETE'] are not read-only"}
3) {'refused': True, 'reason': "statement types ['SELECT', 'DROP'] are not read-only"}
4) read-only connection: InvalidInputException - Invalid Input Error: Cannot execute statement of type "INSERT" on database "course" which 


## Benchmark: 10 multi-hop questions, gold answers from SQL

Gold answers are computed straight from the warehouse with SQL — independently of the graph and of any LLM.

In [4]:
def sql1(q, *p):
    return con.execute(q, list(p)).fetchone()[0]

QUESTIONS = [
    ("q01", "INC0000388 was caused by a change request. Which other incident references that same change request? Give the incident number.",
     sql1("SELECT number FROM linked WHERE rfc = (SELECT rfc FROM linked WHERE number='INC0000388') AND number <> 'INC0000388'")),
    ("q02", "Which assignment group handled the other incident that references the same change request as INC0000388?",
     sql1("SELECT assignment_group FROM linked WHERE rfc = (SELECT rfc FROM linked WHERE number='INC0000388') AND number <> 'INC0000388'")),
    ("q03", "How many incidents in total (including it) are linked to the same problem record as INC0000057?",
     sql1("SELECT count(*) FROM linked WHERE problem_id = (SELECT problem_id FROM linked WHERE number='INC0000057')")),
    ("q04", "Which assignment group handled the most incidents linked to the same problem record as INC0000057?",
     sql1("SELECT assignment_group FROM linked WHERE problem_id = (SELECT problem_id FROM linked WHERE number='INC0000057') GROUP BY 1 ORDER BY count(*) DESC LIMIT 1")),
    ("q05", "How many distinct assignment groups handled incidents linked to the same problem record as INC0001304?",
     sql1("SELECT count(DISTINCT assignment_group) FROM linked WHERE problem_id = (SELECT problem_id FROM linked WHERE number='INC0001304')")),
    ("q06", "Which problem record is linked to the largest number of critical-priority (1 - Critical) incidents?",
     norm(sql1("SELECT problem_id FROM linked WHERE priority LIKE '1%' AND problem_id IS NOT NULL GROUP BY 1 ORDER BY count(*) DESC LIMIT 1"))),
    ("q07", "Of the incidents linked to the same problem record as INC0011654 (including it), how many breached their SLA?",
     sql1("SELECT count(*) FROM linked WHERE NOT made_sla AND problem_id = (SELECT problem_id FROM linked WHERE number='INC0011654')")),
    ("q08", "Which category do most incidents linked to the same problem record as INC0023310 belong to?",
     sql1("SELECT category FROM linked WHERE problem_id = (SELECT problem_id FROM linked WHERE number='INC0023310') GROUP BY 1 ORDER BY count(*) DESC LIMIT 1")),
    ("q09", "INC0027882 shares a change request with another incident. What priority does that other incident have?",
     sql1("SELECT priority FROM linked WHERE rfc = (SELECT rfc FROM linked WHERE number='INC0027882') AND number <> 'INC0027882'")),
    ("q10", "What percentage of ALL incidents in the warehouse that belong to the same category as INC0001131 breached their SLA? Round to one decimal.",
     round(100 * sql1("""SELECT avg(1 - made_sla::INT) FROM incidents.incident_summary WHERE incident IN
            (SELECT DISTINCT number FROM incidents.incident_events WHERE category = (SELECT category FROM linked WHERE number='INC0001131'))"""), 1)),
]
for qid, text, gold in QUESTIONS:
    print(f"{qid}  gold={gold!r:<18} {text}")

def score(final: str, gold) -> bool:
    final = norm(final or "").lower()
    if isinstance(gold, (int, float)):
        nums = [float(x) for x in re.findall(r"\d+(?:\.\d+)?", final.replace(",", ""))]
        return any(abs(x - gold) <= (0.15 if isinstance(gold, float) else 0) for x in nums)
    gold = norm(str(gold)).lower()
    # entity answers: the gold entity must be named, and no *other* entity of the same kind (no hedged ties)
    kind = re.match(r"(problem id|group|category|inc|\d - )", gold)
    if kind:
        pat = {"problem id": r"problem id \d+", "group": r"group \d+", "category": r"category \d+",
               "inc": r"inc\d+", }.get(kind.group(1), r"\d - (?:critical|high|moderate|low)")
        return set(re.findall(pat, final)) == {gold}
    return gold in final

q01  gold='INC0002325'       INC0000388 was caused by a change request. Which other incident references that same change request? Give the incident number.
q02  gold='Group 15'         Which assignment group handled the other incident that references the same change request as INC0000388?
q03  gold=32                 How many incidents in total (including it) are linked to the same problem record as INC0000057?
q04  gold='Group 54'         Which assignment group handled the most incidents linked to the same problem record as INC0000057?
q05  gold=8                  How many distinct assignment groups handled incidents linked to the same problem record as INC0001304?
q06  gold='Problem ID 213'   Which problem record is linked to the largest number of critical-priority (1 - Critical) incidents?
q07  gold=1                  Of the incidents linked to the same problem record as INC0011654 (including it), how many breached their SLA?
q08  gold='Category 19'      Which category do most incid

## (a) ReAct agent over graph tools vs. (b) flat top-10 retrieval (Slides 20, 41)

The agent prompt says *what* is in the graph, not *which tools to call* — the agent chooses each hop itself.
The baseline embeds a one-line text card per incident (all the same facts as the graph) and answers single-shot from
the 10 most similar cards — the "chunks that do not resemble the question" failure of Slide 32.

In [5]:
import os
import numpy as np
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
CHAT_MODEL, EMBED_MODEL = "gpt-4o-mini", "text-embedding-3-small"
FINAL_RE = re.compile(r"FINAL ANSWER:\s*(.+)", re.I)

AGENT_SYSTEM = (
    "You are an SRE analyst. You answer questions about ServiceNow incidents by exploring an operational knowledge "
    "graph with the provided read-only tools. Node types: Incident, Problem, ChangeRequest, AssignmentGroup, Category, "
    "Priority, SLAOutcome. Relation types: LINKED_TO_PROBLEM, RELATED_CHANGE, CAUSED_BY_CHANGE, ASSIGNED_TO, "
    "IN_CATEGORY, HAS_PRIORITY, SLA_OUTCOME. Gather evidence with tools before answering. "
    "End with one line: 'FINAL ANSWER: <short answer>'.")

def run_agent(question, max_steps=12, verbose=False):
    messages = [{"role": "system", "content": AGENT_SYSTEM}, {"role": "user", "content": question}]
    n_calls, tokens = 0, 0
    for step in range(1, max_steps + 1):
        resp = client.chat.completions.create(model=CHAT_MODEL, messages=messages, tools=TOOLS, temperature=0, seed=7)
        tokens += resp.usage.total_tokens
        msg = resp.choices[0].message
        if not msg.tool_calls:
            m = FINAL_RE.search(msg.content or "")
            return (m.group(1).strip() if m else (msg.content or "")), n_calls, tokens
        messages.append(msg.model_dump(exclude_none=True))
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            result = TOOL_IMPL[tc.function.name](**args)
            n_calls += 1
            if verbose and n_calls <= 6:
                print(f"  [step {step}] {tc.function.name}({args})\n           -> {result[:220]}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
        if verbose and n_calls > 6 and step > 1:
            print(f"  ... ({n_calls} tool calls so far)")
    return "(no answer: step budget exhausted)", n_calls, tokens

def card(r):
    return (f"Incident {r['number']} | priority {r['priority']} | {r['category']} | assignment group {r['assignment_group']} | "
            f"problem {norm(r['problem_id'])} | change {r['rfc']} | caused_by {r['caused_by']} | "
            f"{'SLA met' if r['made_sla'] else 'SLA breached'}")

cards = [card(r) for r in records]
card_emb = np.array([e.embedding for e in client.embeddings.create(model=EMBED_MODEL, input=cards).data])
card_emb /= np.linalg.norm(card_emb, axis=1, keepdims=True)

def run_flat_rag(question, k=10):
    q = np.array(client.embeddings.create(model=EMBED_MODEL, input=[question]).data[0].embedding)
    top = np.argsort(-(card_emb @ (q / np.linalg.norm(q))))[:k]
    context = "\n".join(cards[i] for i in top)
    resp = client.chat.completions.create(model=CHAT_MODEL, temperature=0, seed=7, messages=[
        {"role": "system", "content": "Answer the SRE question using only the retrieved incident records. "
                                      "End with one line: 'FINAL ANSWER: <short answer>'."},
        {"role": "user", "content": f"Question: {question}\n\nRetrieved records:\n{context}"}])
    m = FINAL_RE.search(resp.choices[0].message.content or "")
    return (m.group(1).strip() if m else resp.choices[0].message.content), resp.usage.total_tokens

print("Live ReAct trace for q04 (2 hops: incident -> problem -> incidents -> groups):")
ans, calls, toks = run_agent(QUESTIONS[3][1], verbose=True)
print(f"  FINAL: {ans}   (gold: {QUESTIONS[3][2]}, {calls} tool calls, {toks:,} tokens)")

Live ReAct trace for q04 (2 hops: incident -> problem -> incidents -> groups):


  [step 1] get_neighbors({'node_id': 'INC0000057', 'relation': 'LINKED_TO_PROBLEM'})
           -> {"id": "INC0000057", "label": "Incident", "counts_by_relation": {"LINKED_TO_PROBLEM": 1}, "neighbors": [{"relation": "LINKED_TO_PROBLEM", "id": "Problem ID 2", "label": "Problem"}]}


  [step 2] get_neighbors({'node_id': 'Problem ID 2', 'relation': 'ASSIGNED_TO'})
           -> {"id": "Problem ID 2", "label": "Problem", "counts_by_relation": {}, "neighbors": []}


  [step 3] get_neighbors({'node_id': 'Problem ID 2', 'relation': 'LINKED_TO_PROBLEM'})
           -> {"id": "Problem ID 2", "label": "Problem", "counts_by_relation": {"LINKED_TO_PROBLEM": 32}, "neighbors": [{"relation": "LINKED_TO_PROBLEM", "id": "INC0000057", "label": "Incident"}, {"relation": "LINKED_TO_PROBLEM", "id"


  [step 4] get_neighbors({'node_id': 'INC0000073', 'relation': 'ASSIGNED_TO'})
           -> {"id": "INC0000073", "label": "Incident", "counts_by_relation": {"ASSIGNED_TO": 1}, "neighbors": [{"relation": "ASSIGNED_TO", "id": "Group 54", "label": "AssignmentGroup"}]}
  [step 4] get_neighbors({'node_id': 'INC0000083', 'relation': 'ASSIGNED_TO'})
           -> {"id": "INC0000083", "label": "Incident", "counts_by_relation": {"ASSIGNED_TO": 1}, "neighbors": [{"relation": "ASSIGNED_TO", "id": "Group 23", "label": "AssignmentGroup"}]}
  [step 4] get_neighbors({'node_id': 'INC0000085', 'relation': 'ASSIGNED_TO'})
           -> {"id": "INC0000085", "label": "Incident", "counts_by_relation": {"ASSIGNED_TO": 1}, "neighbors": [{"relation": "ASSIGNED_TO", "id": "Group 54", "label": "AssignmentGroup"}]}
  ... (27 tool calls so far)


  FINAL: Group 54   (gold: Group 54, 27 tool calls, 6,774 tokens)


In [6]:
results = []
for qid, text, gold in QUESTIONS:
    a_ans, a_calls, a_tok = run_agent(text)
    b_ans, b_tok = run_flat_rag(text)
    results.append(dict(qid=qid, gold=gold, agent=a_ans, agent_ok=score(a_ans, gold), calls=a_calls, agent_tok=a_tok,
                        flat=b_ans, flat_ok=score(b_ans, gold), flat_tok=b_tok))

print(f"{'q':<4} {'gold':<16} {'ReAct agent (graph tools)':<34} {'ok':<3} {'calls':>5} | {'flat top-10 RAG':<34} {'ok':<3}")
print("-" * 110)
for r in results:
    print(f"{r['qid']:<4} {str(r['gold']):<16} {r['agent'][:33]:<34} {'Y' if r['agent_ok'] else 'n':<3} {r['calls']:>5} | "
          f"{r['flat'][:33]:<34} {'Y' if r['flat_ok'] else 'n':<3}")

q    gold             ReAct agent (graph tools)          ok  calls | flat top-10 RAG                    ok 
--------------------------------------------------------------------------------------------------------------
q01  INC0002325       INC0002325                         Y       2 | None                               n  
q02  Group 15         Group 15                           Y       5 | Unknown                            n  
q03  32               32                                 Y       2 | 4                                  n  
q04  Group 54         Group 54                           Y      27 | Group 54                           Y  
q05  8                8 distinct assignment groups.      Y      24 | 3                                  n  
q06  Problem ID 213   (no answer: step budget exhausted  n      12 | No problem record is linked to mu  n  
q07  1                1                                  Y      14 | 0                                  n  
q08  Category 19      Cat

In [7]:
n = len(results)
agent_acc = sum(r["agent_ok"] for r in results)
flat_acc = sum(r["flat_ok"] for r in results)
avg_calls = sum(r["calls"] for r in results) / n
avg_agent_tok = sum(r["agent_tok"] for r in results) / n
avg_flat_tok = sum(r["flat_tok"] for r in results) / n

print("=" * 72)
print("RESULTS SUMMARY — Notebook 2: ReAct traversal vs. flat retrieval")
print("=" * 72)
print(f"Graph built from warehouse : {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, {len(records)} linked incidents")
print(f"Multi-hop questions        : {n} (gold answers from SQL)")
print(f"ReAct agent accuracy       : {agent_acc}/{n}  ({100*agent_acc/n:.0f}%)")
print(f"Flat top-10 RAG accuracy   : {flat_acc}/{n}  ({100*flat_acc/n:.0f}%)")
print(f"Agent tool calls / question: {avg_calls:.1f}")
print(f"Tokens / question          : agent {avg_agent_tok:,.0f}  vs  flat RAG {avg_flat_tok:,.0f}  ({avg_agent_tok/avg_flat_tok:.1f}x)")
print(f"Write attempts refused     : 2/2 by statement-type guard, + read-only connection as second layer")
print(f"Bug fixed                  : Category 45 = {json.loads(category_sla_stats('Category 45'))['n_incidents']:,} incidents "
      f"(old join reported {buggy:,} event rows)")
print("\nLecture message: the graph answers the relational step exactly (Slide 20); iterative traversal beats one-shot")
print("retrieval on multi-hop questions (Slide 41), at a token cost — and the agent only touches the graph through")
print("templated, read-only tools (Slides 43-44, 61).")

RESULTS SUMMARY — Notebook 2: ReAct traversal vs. flat retrieval
Graph built from warehouse : 893 nodes, 2310 edges, 456 linked incidents
Multi-hop questions        : 10 (gold answers from SQL)
ReAct agent accuracy       : 8/10  (80%)
Flat top-10 RAG accuracy   : 1/10  (10%)
Agent tool calls / question: 10.8
Tokens / question          : agent 4,194  vs  flat RAG 646  (6.5x)
Write attempts refused     : 2/2 by statement-type guard, + read-only connection as second layer
Bug fixed                  : Category 45 = 597 incidents (old join reported 3,271 event rows)

Lecture message: the graph answers the relational step exactly (Slide 20); iterative traversal beats one-shot
retrieval on multi-hop questions (Slide 41), at a token cost — and the agent only touches the graph through
templated, read-only tools (Slides 43-44, 61).
